# ML-bike: Predicting hourly Helsinki city-bike demand

**Goal:** predict how many city-bike trips start in each hour. Operators use this kind of forecast to plan bike redistribution and maintenance.

**Data:** trip records from the Helsinki & Espoo city-bike system (`data/YYYY-MM.csv`), seasons April–October 2022–2025. Each row is one trip with its departure/return time, stations, distance and duration.

**ML problem formulation**
- **Data point:** one hour during the bike season.
- **Features:** calendar information about that hour (hour of day, day of week, month, day of year, weekend and public-holiday flags).
- **Label:** number of trips that start in that hour (a non-negative integer, treated as a real number → regression).
- **Hypothesis spaces:** linear models (on one-hot encoded calendar features) and random forests (tree ensembles).
- **Loss:** squared error for training; MAE, RMSE and R² for evaluation.
- **Validation:** time-based split. Train on 2022–2023, validate on 2024, test on 2025.

Steps:
1. Setup
2. Load the data
3. Clean the data
4. Explore the trips
5. Build the hourly demand dataset
6. Feature engineering
7. Train / validation / test split
8. Train and validate models
9. Final evaluation on the test set
10. Conclusions

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_DIR = Path("data")
RANDOM_STATE = 42

pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 2. Load the data

The monthly files all have the same columns. We only need departure time (to count trips per hour), plus distance and duration (to remove invalid trips), so we load just those three columns to save memory.

In [ ]:
COLUMNS = {
    "Departure": "departure",
    "Covered distance (m)": "distance_m",
    "Duration (sec.)": "duration_s",
}

files = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(files)} files")

frames = []
for f in files:
    month_df = pd.read_csv(
        f,
        usecols=list(COLUMNS),
        dtype={"Covered distance (m)": "float32", "Duration (sec.)": "float32"},
    ).rename(columns=COLUMNS)
    frames.append(month_df)
    print(f"  {f.name}: {len(month_df):>9,} trips")

trips = pd.concat(frames, ignore_index=True)
del frames
# A few rows contain only a date (no time of day); they become NaT and are dropped in step 3.
trips["departure"] = pd.to_datetime(trips["departure"], format="%Y-%m-%dT%H:%M:%S", errors="coerce")

print(f"\nTotal: {len(trips):,} trips")
trips.head()

In [ ]:
trips.info()
trips.describe()

## 3. Clean the data

Following the data provider's guidance, we drop trips that:
- have missing values, including departure timestamps with no time of day (these can't be placed in an hour),
- lasted under 10 seconds or covered under 10 metres (usually false starts or a bike being re-docked),
- lasted over 5 hours (usually a bike returned incorrectly rather than a real ride).

In [ ]:
n_raw = len(trips)

trips = trips.dropna()
n_after_na = len(trips)

valid = (
    (trips["duration_s"] >= 10)
    & (trips["distance_m"] >= 10)
    & (trips["duration_s"] <= 5 * 3600)
)
trips = trips[valid].reset_index(drop=True)

print(f"Raw trips:                 {n_raw:,}")
print(f"Removed (missing values):  {n_raw - n_after_na:,}")
print(f"Removed (invalid trips):   {n_after_na - len(trips):,}")
print(f"Remaining trips:           {len(trips):,} ({len(trips) / n_raw:.1%})")

## 4. Explore the trips

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(trips["duration_s"] / 60, bins=100, range=(0, 60), color="tab:blue")
axes[0].set_title("Trip duration")
axes[0].set_xlabel("minutes")
axes[0].set_ylabel("trips")

axes[1].hist(trips["distance_m"] / 1000, bins=100, range=(0, 10), color="tab:orange")
axes[1].set_title("Trip distance")
axes[1].set_xlabel("km")

plt.tight_layout()
plt.show()

print(f"Median duration: {trips['duration_s'].median() / 60:.1f} min")
print(f"Median distance: {trips['distance_m'].median() / 1000:.2f} km")

In [ ]:
monthly = (
    trips.groupby([trips["departure"].dt.month, trips["departure"].dt.year])
    .size()
    .unstack()
)
monthly.index.name = "month"
monthly.columns.name = "year"

ax = monthly.plot(kind="bar", figsize=(11, 4))
ax.set_title("Trips per month, by year")
ax.set_ylabel("trips")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

monthly

## 5. Build the hourly demand dataset

We count trips per hour. Hours with no departures count as **0**, so the dataset includes quiet night hours too.

Service runs only during the season, and its start and end dates vary by year. So we keep only **service days**: days between each season's first and last day of normal operation. That way the off-season (November–March) is not recorded as millions of zero-demand hours.

In [ ]:
daily = trips.set_index("departure").resample("D").size()
daily = daily[daily.index.month.isin(range(4, 11))]

# A day is in operation if it has at least 1% of that season's median daily trips.
# This skips the few test rides logged before a season properly starts.
threshold = daily.groupby(daily.index.year).transform("median") * 0.01
active = daily[daily >= threshold]
season_bounds = active.index.to_series().groupby(active.index.year).agg(["min", "max"])
season_bounds.index.name = "year"

service_days = pd.DatetimeIndex(
    np.concatenate([pd.date_range(row["min"], row["max"], freq="D") for _, row in season_bounds.iterrows()])
)
season_bounds.assign(days=(season_bounds["max"] - season_bounds["min"]).dt.days + 1)

In [ ]:
hourly = trips.set_index("departure").resample("h").size().rename("trips")

# Complete hourly index over all service days (missing hours become 0 trips).
full_index = pd.DatetimeIndex(
    np.concatenate([pd.date_range(d, periods=24, freq="h") for d in service_days])
)
hourly = hourly.reindex(full_index, fill_value=0).to_frame()
hourly.index.name = "time"

print(f"Hourly data points: {len(hourly):,}")
hourly.describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
daily_totals = hourly["trips"].resample("D").sum()
daily_totals = daily_totals[daily_totals.index.isin(service_days)]
for year, s in daily_totals.groupby(daily_totals.index.year):
    ax.plot(s.index.dayofyear, s.values, label=str(year), lw=1)
ax.set_title("Trips per day over the season")
ax.set_xlabel("day of year")
ax.set_ylabel("trips")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
profile = hourly.assign(
    hour=hourly.index.hour,
    day_type=np.where(hourly.index.dayofweek >= 5, "weekend", "weekday"),
)
ax = profile.pivot_table(index="hour", columns="day_type", values="trips", aggfunc="mean").plot(marker="o")
ax.set_title("Average trips per hour of day")
ax.set_ylabel("trips / hour")
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

**Takeaways:**
- Weekdays show clear commuter peaks around 8 and 16–17. Weekend demand builds slowly to a peak in the afternoon.
- Demand is highest in June–August and lower at the start and end of the season.
- These patterns are why calendar features (hour, weekday/weekend, time of year) should predict demand well.

## 6. Feature engineering

All features come from the timestamp, so they are known in advance for any future hour. Public holidays behave like weekends, so we flag the Finnish public holidays that fall in April–October.

In [ ]:
FI_HOLIDAYS = pd.to_datetime([
    # Good Friday, Easter Monday, May Day, Ascension Day, Midsummer Eve, Midsummer Day
    "2022-04-15", "2022-04-18", "2022-05-01", "2022-05-26", "2022-06-24", "2022-06-25",
    "2023-04-07", "2023-04-10", "2023-05-01", "2023-05-18", "2023-06-23", "2023-06-24",
    "2024-04-01", "2024-05-01", "2024-05-09", "2024-06-21", "2024-06-22",
    "2025-04-18", "2025-04-21", "2025-05-01", "2025-05-29", "2025-06-20", "2025-06-21",
])


def make_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    X = pd.DataFrame(index=index)
    X["hour"] = index.hour
    X["dayofweek"] = index.dayofweek
    X["month"] = index.month
    X["dayofyear"] = index.dayofyear
    X["is_weekend"] = (index.dayofweek >= 5).astype(int)
    X["is_holiday"] = index.normalize().isin(FI_HOLIDAYS).astype(int)
    # Holidays behave like weekends; combine with hour for the linear model.
    day_type = np.where((X["is_weekend"] == 1) | (X["is_holiday"] == 1), "off", "work")
    X["hour_daytype"] = X["hour"].astype(str) + "_" + day_type
    return X


X_all = make_features(hourly.index)
y_all = hourly["trips"]
X_all.head()

## 7. Train / validation / test split

The data is a time series, so a random split would leak information: neighbouring hours are very similar. We split by season instead:

| Set | Seasons | Purpose |
|---|---|---|
| Training | 2022, 2023 | fit model parameters |
| Validation | 2024 | compare models and choose one |
| Test | 2025 | final, unbiased estimate of performance |

In [ ]:
years = X_all.index.year
train_mask = years <= 2023
val_mask = years == 2024
test_mask = years == 2025

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

assert len(X_train) + len(X_val) + len(X_test) == len(X_all)
for name, X in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    print(f"{name:<11} {len(X):>6,} hours  ({X.index.min().date()} -> {X.index.max().date()})")

## 8. Train and validate models

We compare three models:
1. **Baseline:** the mean training demand for each (hour, working/off day) pair. Any real model has to beat this.
2. **Linear regression:** one-hot encoded `hour_daytype`, `month` and `dayofweek`, fitted by minimising squared error. It is simple and easy to interpret, but it can only add up effects, not combine them in more complex ways.
3. **Random forest:** an ensemble of regression trees on the numeric calendar features. It can capture non-linear effects and interactions, e.g. how the hourly profile changes over the season.

In [ ]:
def evaluate(y_true, y_pred) -> dict:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


class HourlyMeanBaseline:
    # Predicts the mean training demand for each (hour, day type) combination.
    def fit(self, X, y):
        self.means_ = y.groupby(X["hour_daytype"]).mean()
        self.global_mean_ = y.mean()
        return self

    def predict(self, X):
        return X["hour_daytype"].map(self.means_).fillna(self.global_mean_).to_numpy()


LINEAR_CATEGORICAL = ["hour_daytype", "month", "dayofweek"]
TREE_FEATURES = ["hour", "dayofweek", "month", "dayofyear", "is_weekend", "is_holiday"]


class ColumnSubset:
    # Wraps a model so it only sees the given columns of X.
    def __init__(self, model, columns):
        self.model, self.columns = model, columns

    def fit(self, X, y):
        self.model.fit(X[self.columns], y)
        return self

    def predict(self, X):
        return self.model.predict(X[self.columns])


def build_models() -> dict:
    return {
        "Baseline (hour x day type mean)": HourlyMeanBaseline(),
        "Linear regression": make_pipeline(
            ColumnTransformer(
                [("onehot", OneHotEncoder(handle_unknown="ignore"), LINEAR_CATEGORICAL)],
                remainder="drop",
            ),
            LinearRegression(),
        ),
        "Random forest": ColumnSubset(
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=5,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
            TREE_FEATURES,
        ),
    }

In [ ]:
models = build_models()
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_scores = evaluate(y_train, np.clip(model.predict(X_train), 0, None))
    val_scores = evaluate(y_val, np.clip(model.predict(X_val), 0, None))
    results.append({
        "model": name,
        **{f"train {k}": v for k, v in train_scores.items()},
        **{f"val {k}": v for k, v in val_scores.items()},
    })

results = pd.DataFrame(results).set_index("model")
results

Predictions are clipped at 0 because trip counts cannot be negative. We choose the final model by **lowest validation MAE**, i.e. the smallest average error in trips per hour.

In [ ]:
best_name = results["val MAE"].idxmin()
print(f"Selected model: {best_name}")

ax = results[["train MAE", "val MAE"]].plot(kind="barh", figsize=(9, 3.5))
ax.set_title("Mean absolute error (trips / hour)")
ax.set_xlabel("MAE")
plt.tight_layout()
plt.show()

## 9. Final evaluation on the test set

We refit the selected model on the training **and** validation seasons (2022–2024), then evaluate it once on the untouched 2025 season.

In [ ]:
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

final_model = build_models()[best_name].fit(X_trainval, y_trainval)
y_test_pred = np.clip(final_model.predict(X_test), 0, None)

test_scores = pd.Series(evaluate(y_test, y_test_pred), name=best_name)
print(f"Test performance (2025 season), {best_name}:")
test_scores.to_frame().T

In [ ]:
# Actual vs predicted for one week in mid-season 2025.
week = slice("2025-06-30", "2025-07-06")
pred_series = pd.Series(y_test_pred, index=X_test.index)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_test.loc[week].index, y_test.loc[week].values, label="actual", lw=1.5)
ax.plot(pred_series.loc[week].index, pred_series.loc[week].values, label="predicted", lw=1.5, ls="--")
ax.set_title(f"Hourly trips, week of 30 June 2025 ({best_name})")
ax.set_ylabel("trips / hour")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, y_test_pred, s=3, alpha=0.3)
lim = max(y_test.max(), y_test_pred.max())
axes[0].plot([0, lim], [0, lim], color="black", lw=1)
axes[0].set_title("Predicted vs actual (test)")
axes[0].set_xlabel("actual trips / hour")
axes[0].set_ylabel("predicted trips / hour")

residuals = y_test - y_test_pred
residuals.groupby(X_test["month"]).mean().plot(kind="bar", ax=axes[1], color="tab:red")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_title("Mean residual (actual - predicted) by month")
axes[1].set_xlabel("month")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Which features does the random forest rely on? (Fitted on 2022–2024.)
rf = build_models()["Random forest"].fit(X_trainval, y_trainval)
importances = pd.Series(rf.model.feature_importances_, index=TREE_FEATURES).sort_values()

ax = importances.plot(kind="barh", figsize=(8, 3.5), color="tab:green")
ax.set_title("Random forest feature importance")
plt.tight_layout()
plt.show()

## 10. Conclusions

- Calendar features explain most of the variation in hourly demand: the daily commuter and leisure patterns, and the seasonal curve.
- The validation table (step 8) compares the models. We chose the one with the lowest validation MAE and reported its error on the unseen 2025 season in step 9. Because the test set was not used for training or model selection, this is a fair estimate of future performance.
- The monthly residual plot shows what calendar features miss. Overall usage changes from year to year, and calendar features cannot capture that.

**Limitations and next steps**
- **Weather** (rain, temperature) strongly affects cycling but is not in this dataset. Adding FMI weather observations is the most promising improvement.
- **Year-level trends** (fleet size, number of stations, pricing) are not modelled. A tree model cannot extrapolate a trend, so a yearly scaling factor or recent-demand features (e.g. demand at the same hour last week) could help.
- **Station-level forecasts** would be more useful for bike redistribution than a single system-wide total.